# DEMI v17 worked example (sertraline response, as in the article)

This notebook follows the manuscript example: anxiety → citalopram non-response → major depressive disorder (MDD) → sertraline response. It shows temporal scores, total and direct effects, the correction factor, and the predicted probability.

**Setup:** keep this notebook in `paper/` next to `demi_v17.py`, install NumPy, pandas, and IPython, and set `DEMI_DB_PATH` to an authorized copy of the matching study database. Run cells in order. The database and All of Us-derived relationship tables are not distributed with the code.

The manuscript computes temporal scores from these three features, including singleton contributions. The main core prefers global database scores; its local fallback uses precedence counts without singleton contributions. These procedures can produce different orders. The final section compares them explicitly and checks the core model separately.

The example uses prior odds of **1,000/4,000 = 0.25** (probability 0.20) for the manuscript's case–control sample. This is not an estimate of population response prevalence. Outputs are displayed locally; CSV export is off by default. The accompanying code license does not grant rights to database contents or derived outputs.

In [ ]:
from pathlib import Path
import importlib.util
import math
import os
import sqlite3
import warnings

import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    # Plain-Python execution can still display tables as text.
    def display(frame):
        print(frame.to_string(index=False) if isinstance(frame, pd.DataFrame) else frame)

DB_PATH = Path(os.environ['DEMI_DB_PATH'])  # authorized study database; not distributed
CORE_PATH = Path(os.getenv('DEMI_CORE_FILE', 'demi_v17.py')).resolve()
EDGE_TABLE = os.getenv('DEMI_EDGE_TABLE', 'demi_counts_published')
CONCEPT_TABLE = os.getenv('DEMI_CONCEPT_TABLE', 'concept_names')

# Confirm these four codes against the concept-resolution table below.
CONCEPT_CODES = {
    'MDD': os.getenv('DEMI_EXAMPLE_MDD_CODE', 'dx_SNOMED_370143000'),
    'ANX': os.getenv('DEMI_EXAMPLE_ANXIETY_CODE', 'dx_SNOMED_48694002'),
    'CIT': os.getenv('DEMI_EXAMPLE_CITALOPRAM_FAIL_CODE', 'aa_meas_citalopram_fail'),
    'Y': os.getenv('DEMI_EXAMPLE_SERTRALINE_RESPONSE_CODE', 'aa_meas_sertraline_rem'),
}
DISPLAY_NAMES = {
    'MDD': 'major depressive disorder',
    'ANX': 'anxiety',
    'CIT': 'citalopram non-response',
    'Y': 'sertraline response',
}

# Manuscript prediction sample: 1,000 responders and 4,000 non-responders.
PRIOR_POSITIVE = 1000
PRIOR_NEGATIVE = 4000
WRITE_OUTPUTS = False  # Enable only inside an authorized research environment.
OUTPUT_DIR = Path(os.getenv('DEMI_EXAMPLE_OUTPUT_DIR', 'demi_example_output')).resolve()
LAMBDA_MIN_COUNT = 15  # Matches the core's default mediation threshold.
EFFECT_RTOL = 1e-6
EFFECT_ATOL = 1e-8


def save_table(frame, filename):
    if WRITE_OUTPUTS:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        frame.to_csv(OUTPUT_DIR / filename, index=False)

if not DB_PATH.is_file():
    raise FileNotFoundError(f'DEMI database not found: {DB_PATH}')
if not CORE_PATH.is_file():
    raise FileNotFoundError(
        f'DEMI module not found: {CORE_PATH}. Keep this notebook in paper/ '
        'next to demi_v17.py, or set DEMI_CORE_FILE.'
    )

spec = importlib.util.spec_from_file_location('demi_manuscript_core', CORE_PATH)
if spec is None or spec.loader is None:
    raise ImportError(f'Could not import DEMI core from {CORE_PATH}')
demi = importlib.util.module_from_spec(spec)
spec.loader.exec_module(demi)

print('Database:', DB_PATH)
print('Core:', CORE_PATH)
print('Configured codes:', CONCEPT_CODES)
print('CSV export enabled:', WRITE_OUTPUTS)

## 1. Resolve the four concepts

Codes must resolve uniquely. If a code is absent, inspect the displayed candidates and correct the configuration; the notebook does not substitute concepts automatically.

In [ ]:
def connect_ro():
    return sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)

def quote_identifier(value):
    if not value.replace('_', '').isalnum():
        raise ValueError(f'Unsafe SQLite identifier: {value!r}')
    return value

edge_table = quote_identifier(EDGE_TABLE)
concept_table = quote_identifier(CONCEPT_TABLE)

with connect_ro() as conn:
    concept_cols = [r[1] for r in conn.execute(f'PRAGMA table_info({concept_table})')]

name_cols = [c for c in ('concept_name', 'standard_concept_name', 'name') if c in concept_cols]
select_cols = ['concept_code_int', 'concept_code'] + name_cols
placeholders = ','.join(['?'] * len(CONCEPT_CODES))
with connect_ro() as conn:
    resolved = pd.read_sql_query(
        f"SELECT {', '.join(select_cols)} FROM {concept_table} "
        f"WHERE concept_code IN ({placeholders})",
        conn,
        params=list(CONCEPT_CODES.values()),
    )

code_to_role = {code: role for role, code in CONCEPT_CODES.items()}
resolved['role'] = resolved['concept_code'].map(code_to_role)
resolved['manuscript_label'] = resolved['role'].map(DISPLAY_NAMES)
resolved = resolved.sort_values('role').reset_index(drop=True)
display(resolved[['role', 'manuscript_label'] + select_cols])

missing_roles = [role for role, code in CONCEPT_CODES.items() if code not in set(resolved['concept_code'])]
if missing_roles:
    search_terms = {
        'MDD': ['major', 'depress'],
        'ANX': ['anxiety'],
        'CIT': ['citalopram'],
        'Y': ['sertraline'],
    }
    text_cols = ['concept_code'] + name_cols
    candidate_frames = []
    with connect_ro() as conn:
        for role in missing_roles:
            clauses = []
            params = []
            for term in search_terms[role]:
                clauses.append('(' + ' OR '.join(
                    [f'LOWER(CAST({c} AS TEXT)) LIKE ?' for c in text_cols]
                ) + ')')
                params.extend([f'%{term.lower()}%'] * len(text_cols))
            q = (f"SELECT {', '.join(select_cols)} FROM {concept_table} "
                 f"WHERE {' AND '.join(clauses)} LIMIT 50")
            cand = pd.read_sql_query(q, conn, params=params)
            cand.insert(0, 'missing_role', role)
            candidate_frames.append(cand)
    candidates = pd.concat(candidate_frames, ignore_index=True) if candidate_frames else pd.DataFrame()
    display(candidates)
    raise KeyError(
        f'Configured codes missing for roles {missing_roles}. Choose the intended code(s) '
        'from the candidate table, update CONCEPT_CODES, and rerun from the configuration cell.'
    )

if resolved['role'].duplicated().any() or len(resolved) != 4:
    raise ValueError('Concept resolution was not one-to-one. Review duplicate concept mappings.')

ROLE_TO_INT = resolved.set_index('role')['concept_code_int'].astype(int).to_dict()
INT_TO_ROLE = {value: key for key, value in ROLE_TO_INT.items()}
ROLE_TO_CODE = resolved.set_index('role')['concept_code'].to_dict()
ROLE_TO_NAME = DISPLAY_NAMES.copy()
print('Resolved integer IDs:', ROLE_TO_INT)

## 2. Read pairwise counts

For `(first, second)`, orient the counts as `a=both`, `b=first only`, `c=second only`, and `d=neither`. The requested database direction is required. Reverse-direction rows are not substituted, and conflicting duplicate rows stop execution.

For temporal scoring, the requested target columns are MDD for MDD–citalopram and MDD–anxiety, and anxiety for anxiety–citalopram. Outcome effects require sertraline response as target. Mediation requires the downstream feature as target.

In [ ]:
NUMERIC_COLUMNS = [
    'target_concept_code_int', 'concept_code_int',
    'n_code_target', 'n_code_no_target', 'n_target_no_code',
    'n_no_code_no_target', 'n_target_before_code',
    'n_code_before_target', 'n_code', 'n_target', 'n_no_target',
    'total_effects', 'total_effects_log',
]
ids = sorted(ROLE_TO_INT.values())
ph = ','.join(['?'] * len(ids))
with connect_ro() as conn:
    edges = pd.read_sql_query(
        f'SELECT * FROM {edge_table} '
        f'WHERE target_concept_code_int IN ({ph}) AND concept_code_int IN ({ph})',
        conn,
        params=ids + ids,
    )
for col in NUMERIC_COLUMNS:
    if col in edges.columns:
        edges[col] = pd.to_numeric(edges[col], errors='coerce')

required_count_cols = [
    'n_code_target', 'n_code_no_target', 'n_target_no_code',
    'n_no_code_no_target', 'n_target_before_code', 'n_code_before_target',
]
missing_count_cols = [c for c in required_count_cols if c not in edges.columns]
if missing_count_cols:
    raise KeyError(f'{EDGE_TABLE} is missing required columns: {missing_count_cols}')

def select_pair_row(first_int, second_int, prefer_target):
    if prefer_target not in (first_int, second_int):
        raise ValueError('An explicit target direction is required.')
    predictor = second_int if prefer_target == first_int else first_int
    pair = edges[
        (edges['target_concept_code_int'] == prefer_target)
        & (edges['concept_code_int'] == predictor)
    ].copy()
    if pair.empty:
        raise KeyError(
            f'Missing directed row: target={INT_TO_ROLE[prefer_target]}, '
            f'predictor={INT_TO_ROLE[predictor]}. Reverse rows are not substituted.'
        )
    check_cols = required_count_cols + [
        col for col in ('total_effects', 'total_effects_log') if col in pair.columns
    ]
    if len(pair[check_cols].drop_duplicates()) != 1:
        raise ValueError(
            f'Conflicting directed rows: target={INT_TO_ROLE[prefer_target]}, '
            f'predictor={INT_TO_ROLE[predictor]}. Verify the database snapshot.'
        )
    row = pair.iloc[0]
    counts = row[required_count_cols].to_numpy(dtype=float)
    if not np.isfinite(counts).all() or (counts < 0).any():
        raise ValueError('Required pairwise counts must be finite and nonnegative.')
    if counts[:4].sum() <= 0:
        raise ValueError('The pairwise denominator must be positive.')
    return row


def orient_pair(first_role, second_role, prefer_target_role):
    first_int = ROLE_TO_INT[first_role]
    second_int = ROLE_TO_INT[second_role]
    prefer_target = ROLE_TO_INT[prefer_target_role]
    row = select_pair_row(first_int, second_int, prefer_target=prefer_target)
    target_is_first = int(row['target_concept_code_int']) == first_int
    a = float(row['n_code_target'])
    d = float(row['n_no_code_no_target'])
    if target_is_first:
        b = float(row['n_target_no_code'])
        c = float(row['n_code_no_target'])
        first_before_second = float(row['n_target_before_code'])
        second_before_first = float(row['n_code_before_target'])
    else:
        b = float(row['n_code_no_target'])
        c = float(row['n_target_no_code'])
        first_before_second = float(row['n_code_before_target'])
        second_before_first = float(row['n_target_before_code'])
    N = a + b + c + d
    out = {
        'first_role': first_role, 'second_role': second_role,
        'first_code': ROLE_TO_CODE[first_role], 'second_code': ROLE_TO_CODE[second_role],
        'source_target_role': INT_TO_ROLE[int(row['target_concept_code_int'])],
        'a_both': a, 'b_first_only': b, 'c_second_only': c, 'd_neither': d,
        'first_before_second': first_before_second,
        'second_before_first': second_before_first,
        'N': N, 'p_first': (a + b) / N, 'p_second': (a + c) / N,
    }
    for col in ('total_effects', 'total_effects_log'):
        out[f'db_{col}'] = float(row[col]) if col in row.index and pd.notna(row[col]) else np.nan
    return out

PAIR = {
    ('MDD', 'CIT'): orient_pair('MDD', 'CIT', 'MDD'),
    ('MDD', 'ANX'): orient_pair('MDD', 'ANX', 'MDD'),
    ('ANX', 'CIT'): orient_pair('ANX', 'CIT', 'ANX'),
    ('MDD', 'Y'): orient_pair('MDD', 'Y', 'Y'),
    ('ANX', 'Y'): orient_pair('ANX', 'Y', 'Y'),
    ('CIT', 'Y'): orient_pair('CIT', 'Y', 'Y'),
}
pair_audit_df = pd.DataFrame(PAIR.values())
display(pair_audit_df)
save_table(pair_audit_df, 'manuscript_example_pair_audit.csv')

## 3. Manuscript temporal order

For each feature pair, add `(first-before-second) − (second-before-first) + (first only) − (second only)` to the first feature's score, and subtract it from the second's score. Sort in descending order and place the outcome last. The expected predictor order is checked; sertraline response is explicitly placed last as the designated outcome. This is the manuscript's local worked-example procedure; global core ordering is compared later.

In [12]:
feature_roles = ['MDD', 'ANX', 'CIT']
feature_pairs = [('MDD', 'CIT'), ('MDD', 'ANX'), ('ANX', 'CIT')]
temporal_scores = {role: 0.0 for role in feature_roles}
temporal_rows = []
for first, second in feature_pairs:
    p = PAIR[(first, second)]
    contribution = (
        p['first_before_second'] - p['second_before_first']
        + p['b_first_only'] - p['c_second_only']
    )
    temporal_scores[first] += contribution
    temporal_scores[second] -= contribution
    temporal_rows.append({
        'first': first, 'second': second,
        'first_before_second': p['first_before_second'],
        'second_before_first': p['second_before_first'],
        'first_only': p['b_first_only'], 'second_only': p['c_second_only'],
        'score_contribution_to_first': contribution,
    })

temporal_order_roles = sorted(
    feature_roles, key=lambda role: (-temporal_scores[role], ROLE_TO_INT[role])
)
expected_temporal_order_roles = ['ANX', 'CIT', 'MDD']
if temporal_order_roles != expected_temporal_order_roles:
    observed = ' -> '.join(ROLE_TO_NAME[r] for r in temporal_order_roles)
    expected = ' -> '.join(ROLE_TO_NAME[r] for r in expected_temporal_order_roles)
    raise ValueError(
        f'The database-derived order is {observed}, but this manuscript example expects '
        f'{expected}. Confirm the database version and concept codes before reporting values.'
    )
temporal_df = pd.DataFrame({
    'role': feature_roles,
    'name': [ROLE_TO_NAME[r] for r in feature_roles],
    'score': [temporal_scores[r] for r in feature_roles],
}).sort_values(['score', 'role'], ascending=[False, True])
display(pd.DataFrame(temporal_rows))
display(temporal_df)
print('Temporal order:', ' -> '.join([ROLE_TO_NAME[r] for r in temporal_order_roles] + [ROLE_TO_NAME['Y']]))

,role,name,score
1,ANX,anxiety,90886.0
2,CIT,citalopram non-response,-7698.0
0,MDD,major depressive disorder,-83188.0


Temporal order: anxiety -> citalopram non-response -> major depressive disorder -> sertraline response


## 4. Total effects and same-sign recursion

Recompute adjusted odds ratios from counts and check their log effects against the stored `total_effects_log` used by the core. A disagreement stops execution; confirm the database snapshot rather than silently switching formulas.

Within each sign group, process features backward. Subtract the indirect contribution `(conditional probability − downstream prevalence) × downstream direct effect`, applying the core's minimum-count and nonzero-weight rules. Marginal prevalences come from each feature–outcome pair's own denominator. The case–control prior is specified separately.

In [13]:
total_effect = {}
total_rows = []
prevalence = {}
for role in feature_roles:
    p = PAIR[(role, 'Y')]
    a, b, c, d = p['a_both'], p['b_first_only'], p['c_second_only'], p['d_neither']
    adjusted_or = float(demi.compute_or_v17_scalar(a, b, c, d))
    if not np.isfinite(adjusted_or) or adjusted_or <= 0:
        raise ValueError(f'Undefined adjusted OR for {role}: a={a}, b={b}, c={c}, d={d}')
    recomputed_T = math.log(adjusted_or)
    T = p['db_total_effects_log']
    if not np.isfinite(T):
        raise ValueError(f'Missing stored total_effects_log for {role}.')
    if not np.isclose(T, recomputed_T, rtol=EFFECT_RTOL, atol=EFFECT_ATOL):
        raise ValueError(
            f'Total-effect mismatch for {role}: stored={T}, '
            f'recomputed={recomputed_T}. Verify the database snapshot.'
        )
    total_effect[role] = T
    prevalence[role] = p['p_first']
    total_rows.append({
        'role': role, 'name': ROLE_TO_NAME[role],
        'a': a, 'b': b, 'c': c, 'd': d,
        'raw_OR': (a * d) / (b * c) if b > 0 and c > 0 else np.nan,
        'adjusted_OR': adjusted_or, 'recomputed_T_log_OR': recomputed_T, 'T_log_OR': T,
        'group': 'K+' if T > 0 else 'K-' if T < 0 else 'excluded',
        'prevalence': prevalence[role],
        'db_total_effects': p['db_total_effects'],
        'db_total_effects_log': p['db_total_effects_log'],
    })
total_df = pd.DataFrame(total_rows)
display(total_df)

groups = {
    'K+': [r for r in temporal_order_roles if total_effect[r] > 0],
    'K-': [r for r in temporal_order_roles if total_effect[r] < 0],
}
expected_groups = {'K+': ['ANX', 'MDD'], 'K-': ['CIT']}
if groups != expected_groups:
    raise ValueError(
        f'Observed sign-specific groups {groups}, but the manuscript example expects '
        f'{expected_groups}. Confirm the database version and concept codes.'
    )
direct_effect = {}
mediation_rows = []
for group_name, group in groups.items():
    sign = 1 if group_name == 'K+' else -1
    for i in range(len(group) - 1, -1, -1):
        upstream = group[i]
        indirect = 0.0
        for downstream in group[i + 1:]:
            pair = orient_pair(upstream, downstream, downstream)
            upstream_count = pair['a_both'] + pair['b_first_only']
            lam = (
                pair['a_both'] / upstream_count
                if upstream_count >= LAMBDA_MIN_COUNT and upstream_count > 0 else 0.0
            )
            p_down = prevalence[downstream]
            excess = lam - p_down if lam != 0.0 else 0.0
            component = excess * direct_effect[downstream]
            indirect += component
            mediation_rows.append({
                'group': group_name, 'upstream': upstream, 'downstream': downstream,
                'lambda': lam, 'p_downstream': p_down, 'excess': excess,
                'D_downstream': direct_effect[downstream],
                'indirect_component': component,
            })
        raw_direct = total_effect[upstream] - indirect
        direct_effect[upstream] = max(0.0, raw_direct) if sign == 1 else min(0.0, raw_direct)

mediation_df = pd.DataFrame(mediation_rows)
direct_df = total_df[['role', 'name', 'T_log_OR', 'group', 'prevalence']].copy()
direct_df['D_direct_log_OR'] = direct_df['role'].map(direct_effect)
display(mediation_df)
display(direct_df)
save_table(direct_df, 'manuscript_example_direct_effects.csv')

core_weights = {role: pd.Series(dtype=float) for role in feature_roles}
for row in mediation_rows:
    core_weights[row['upstream']].loc[row['downstream']] = row['lambda']
core_direct = {}
for group_name, group in groups.items():
    core_direct.update(demi._run_backward_recursion(
        group, pd.Series(total_effect), core_weights,
        sign=1 if group_name == 'K+' else -1,
        predictor_prevalence=pd.Series(prevalence),
    ).to_dict())
for role in feature_roles:
    np.testing.assert_allclose(direct_effect[role], core_direct[role], rtol=1e-12, atol=1e-12)
print('PASS: step-by-step direct effects match the core recursion for these inputs.')

,role,name,T_log_OR,group,prevalence,D_direct_log_OR
0,MDD,major depressive disorder,0.268586,K+,0.680387,0.268586
1,ANX,anxiety,0.338394,K+,0.780564,0.376055
2,CIT,citalopram non-response,-0.630566,K-,0.670244,-0.630566


PASS: step-by-step direct effects match the core recursion for these inputs.


## 5. Correction factor and prediction

Calculate `log(Lambda) = sum(log(1 − p_k + p_k × exp(D_k)))` over the three features. For a patient presenting all three features, `log(posterior odds) = log(0.25) + sum(D_k) − log(Lambda)`. Evaluate probability on the log scale for numerical stability. The prior applies to the manuscript's sampled prediction setting.

In [14]:
log_lambda = float(demi._compute_log_correction_factor(
    pd.Series(prevalence, dtype=float), pd.Series(direct_effect, dtype=float),
))
prior_odds = PRIOR_POSITIVE / PRIOR_NEGATIVE
if not np.isfinite(prior_odds) or prior_odds <= 0:
    raise ValueError('Prior odds must be finite and positive.')
log_posterior_odds = math.log(prior_odds) + sum(direct_effect.values()) - log_lambda
if not np.isfinite(log_lambda) or not np.isfinite(log_posterior_odds):
    raise ValueError('The correction factor and posterior log odds must be finite.')
p_hat = float(np.exp(-np.logaddexp(0.0, -log_posterior_odds)))
# Exp-scale values may exceed floating-point range even with a finite log value.
Lambda = float(np.exp(log_lambda)) if log_lambda <= math.log(np.finfo(float).max) else np.inf
phi = float(np.exp(log_posterior_odds)) if log_posterior_odds <= math.log(np.finfo(float).max) else np.inf
prediction_df = pd.DataFrame([{
    'prior_odds': prior_odds,
    'sample_prior_probability': prior_odds / (1 + prior_odds),
    'log_correction_factor': log_lambda,
    'correction_factor': Lambda,
    'posterior_log_odds': log_posterior_odds,
    'posterior_odds': phi,
    'predicted_probability': p_hat,
}])
display(prediction_df)
save_table(prediction_df, 'manuscript_example_prediction.csv')

,prior_odds,sample_prior_probability,log_correction_factor,correction_factor,posterior_log_odds,posterior_odds,predicted_probability
0,0.25,0.2,0.118997,1.126366,-1.491216,0.225099,0.183739


## 6. Compare with the main core pipeline

This audit calls `run_demi` on the same three candidate features, using stored total effects, the same sample prior, and the core's default hierarchy filtering and published temporal ordering. `user_cap=6` leaves room for all three features across the two selection pools; the usual EPV rule still applies.

The manuscript example fixes three patient features and uses local scores with singleton contributions. The core can filter or select different features, uses global scores when available, and computes each feature's prevalence using its own predictor–outcome denominator. Therefore numerical equality is **checked and reported, not assumed**. The same-input recursion check above isolates the mathematical recursion from those pipeline differences. Any difference here requires review before claiming exact end-to-end agreement.

The stored correction-factor table is optional. If it or the target row is absent, the core still computes the model correction factor and completes this comparison; stored audit values are unavailable.

In [15]:
event_ids = [ROLE_TO_INT[role] for role in feature_roles] + [ROLE_TO_INT['Y']]
core_edges = edges[
    edges['target_concept_code_int'].isin(event_ids)
    & edges['concept_code_int'].isin([ROLE_TO_INT[r] for r in feature_roles])
].copy()
# The stored correction-factor table is used only for the core audit.
with connect_ro() as conn:
    table_names = {row[0] for row in conn.execute("SELECT name FROM sqlite_master WHERE type='table'")}
correction_table = os.getenv('DEMI_CORRECTION_FACTOR_TABLE', 'demi_correction_factors')
if correction_table not in table_names:
    print('No stored correction-factor audit table; continuing with the '
          'correction factor calculated from the current model.')
core_result = demi.run_demi(
    core_edges,
    force_outcome=ROLE_TO_INT['Y'],
    db_path=str(DB_PATH),
    pref_override=prior_odds / (1 + prior_odds),
    lambda_min_count=LAMBDA_MIN_COUNT,
    epv=5,
    user_cap=6,
    correction_factor_table_name=correction_table,
)

comparison_rows = []
for role in feature_roles:
    concept_id = ROLE_TO_INT[role]
    selected = concept_id in core_result['direct_log_or'].index
    core_d = float(core_result['direct_log_or'].get(concept_id, np.nan))
    core_p = float(core_result['predictor_prevalence'].get(concept_id, np.nan))
    comparison_rows.append({
        'feature': ROLE_TO_NAME[role],
        'selected_by_core': selected,
        'worked_direct_effect': direct_effect[role],
        'core_direct_effect': core_d,
        'direct_effect_matches': bool(np.isclose(direct_effect[role], core_d, rtol=EFFECT_RTOL, atol=EFFECT_ATOL)),
        'worked_prevalence': prevalence[role],
        'core_prevalence': core_p,
        'prevalence_matches': bool(np.isclose(prevalence[role], core_p, rtol=EFFECT_RTOL, atol=EFFECT_ATOL)),
    })
comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)
core_order = [INT_TO_ROLE[int(x)] for x in core_result['temporal_order'] if int(x) != ROLE_TO_INT['Y']]
order_matches = core_order == temporal_order_roles
patient = {ROLE_TO_INT[r]: 1.0 for r in feature_roles}
core_probability = float(core_result['predict_proba'](patient))
probability_matches = bool(np.isclose(p_hat, core_probability, rtol=EFFECT_RTOL, atol=EFFECT_ATOL))
pipeline_check_df = pd.DataFrame([{
    'worked_order': ' -> '.join(temporal_order_roles),
    'core_order': ' -> '.join(core_order),
    'order_matches': order_matches,
    'worked_probability': p_hat,
    'core_probability': core_probability,
    'probability_matches': probability_matches,
}])
display(pipeline_check_df)
if not (order_matches and probability_matches and comparison_df['selected_by_core'].all()
        and comparison_df['direct_effect_matches'].all() and comparison_df['prevalence_matches'].all()):
    warnings.warn(
        'The manuscript worked example and the main core pipeline differ. '
        'Review ordering, selection, hierarchy exclusions, and prevalence denominators '
        'in the displayed audit before claiming exact agreement.'
    )
save_table(comparison_df, 'manuscript_example_core_comparison.csv')
save_table(pipeline_check_df, 'manuscript_example_pipeline_check.csv')
print('CSV export enabled:', WRITE_OUTPUTS)

,feature,selected_by_core,worked_direct_effect,core_direct_effect,direct_effect_matches,worked_prevalence,core_prevalence,prevalence_matches
0,major depressive disorder,True,0.268586,0.268586,True,0.680387,0.680387,True
1,anxiety,True,0.376055,0.376055,True,0.780564,0.780564,True
2,citalopram non-response,True,-0.630566,-0.630566,True,0.670244,0.670244,True


,worked_order,core_order,order_matches,worked_probability,core_probability,probability_matches
0,ANX -> CIT -> MDD,ANX -> MDD -> CIT,False,0.183739,0.183739,True
